# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one (client, content item, report_date) -- a content-item-day** -- sourced from
`fact_content_daily_performance`, joined to `dim_content` (static page attributes) and
`dim_clients` (per-client history/availability windows).

**Time window for everything below: `month=2026-03`.** That's a mid-panel month, not the final
month -- the brief is explicit that `_sample` / the final month (June 2026) is a sealed test
month and must never be used to develop label logic, only to test query mechanics. March gives
a normal, representative slice with no outcome-window contamination.

Connecting first and confirming table sizes (metadata-only, near-free) before touching any row.


In [1]:
# %pip -q install duckdb huggingface_hub

In [2]:

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    # single month partition for iteration -- NOT the full 79M-row table, NOT the final-month sample
    'fact_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:12} {n:>12,} rows')

# Confirm the real column names before locking in the contract below --
# starter-CSV column names (content_type, main_intent, word_count) are documented for the CSV,
# not guaranteed identical in dim_content, so check rather than assume.
print()
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()['column_name'].tolist())
print()
print("fact_content_daily_performance (March) columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']}").df()['column_name'].tolist())


dim_clients           104 rows
dim_content       519,606 rows
fact_march      9,841,378 rows

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_content_daily_performance (March) columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', '

## 2. Fields: feature / label / context / excluded

**Table(s) used:** `fact_content_daily_performance` (month=2026-03 partition), joined to
`dim_content` on `content_hash_id`, joined to `dim_clients` on `client_hash_id`.

**Label / proxy:** daily CTR, `gsc_clicks / gsc_impressions`, for that content-day. This is the
thing the CTR/Engagement Opportunity lane scores pages against -- never a feature.

- **Feature** (knowable before the day's clicks happen):
  - `gsc_avg_position` for that day -- Google's placement, set independently of that day's clicks.
  - `dim_content` static attributes (content type / intent / word-count equivalents -- exact
    column names confirmed by the `DESCRIBE` above, not assumed) -- set at publish time.
  - content age at `report_date` (from `dim_content`'s creation timestamp) -- computable from a
    date that predates the report day by construction.
  - a trailing window feature built ONLY from days strictly before the current `report_date`
    (e.g. prior-7-day average position) -- knowable because it excludes the day itself.
- **Context** (grouping/joining only, never model input): `client_hash_id`, `content_hash_id`,
  `report_date`.
- **Excluded:**
  - `gsc_clicks` -- it's the numerator of the label itself; including it as a "feature" is the
    trap in Section 3.
  - GA4 engagement columns on rows where `ga4_data_available = FALSE` -- zero-filled placeholders
    before a client's GA4 start, not real zero-engagement observations.
  - `keyword_hash_id` / `url_hash_id` -- scrambled join keys with no real-world meaning; useful
    for grouping/leakage checks, never as model input.


In [3]:
# No computation needed here -- this cell documents the classification above
# so it lives next to code, per the contract pattern.
feature_cols_plan = ["gsc_avg_position", "<dim_content static attrs, confirmed above>", "content_age_days", "prior_7d_avg_position"]
context_cols = ["client_hash_id", "content_hash_id", "report_date"]
excluded_cols = ["gsc_clicks (label numerator)", "GA4 cols where ga4_data_available = FALSE", "keyword_hash_id", "url_hash_id"]
label_col = "ctr = gsc_clicks / gsc_impressions"

print("label:", label_col)
print("planned features:", feature_cols_plan)
print("context only:", context_cols)
print("excluded:", excluded_cols)


label: ctr = gsc_clicks / gsc_impressions
planned features: ['gsc_avg_position', '<dim_content static attrs, confirmed above>', 'content_age_days', 'prior_7d_avg_position']
context only: ['client_hash_id', 'content_hash_id', 'report_date']
excluded: ['gsc_clicks (label numerator)', 'GA4 cols where ga4_data_available = FALSE', 'keyword_hash_id', 'url_hash_id']


## 3. Verify it with queries (grain, counts, availability)

Three queries, each checking one claim above -- on the March 2026 partition only.


In [4]:
# Query 1 -- GRAIN: one row really is one (client, content, day). Zero rows back = grain holds.
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate (client, content, day) combos found: {len(grain_check)}  (expect 0)")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, day) combos found: 0  (expect 0)


,client_hash_id,content_hash_id,report_date,c


In [5]:
# Query 2 -- ROW COUNT + DATE SPAN for the March slice
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_march']}
""").df()
counts


,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [6]:
# Query 3 -- AVAILABILITY: how many March rows have real GA4 engagement data vs zero-filled placeholders
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_march']}
""").df()
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


## 4. Five features (max) -- built from the same March slice

Each feature gets one line: knowable at the decision moment because...


In [7]:
feature_frame = con.sql(f"""
    WITH ranked AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            f.gsc_avg_position,
            f.gsc_impressions,
            f.gsc_clicks,
            -- prior-7-day average position, using ONLY days strictly before this report_date
            AVG(f.gsc_avg_position) OVER (
                PARTITION BY f.client_hash_id, f.content_hash_id
                ORDER BY f.report_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS prior_7d_avg_position
        FROM {TABLES['fact_march']} f
        WHERE f.gsc_impressions >= 5  -- drop near-zero-impression days, too noisy to score
    )
    SELECT r.*, d.* EXCLUDE (content_hash_id)
    FROM ranked r
    LEFT JOIN {TABLES['dim_content']} d USING (content_hash_id)
""").df()

print(f"{len(feature_frame):,} content-days in the feature frame")
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2,637,949 content-days in the feature frame


,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,prior_7d_avg_position,client_hash_id_1,keyword_hash_id,url_hash_id,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_fef1a8f436438636,content_0081d123d9e489eb,2026-03-01,7.812500,16,0,NaN,client_fef1a8f436438636,keyword_b8e40434e7f63eab,url_ad7808313ab6a14c,...,3,2025-12-26,None,gpt-4o-mini,9519,1567,NaT,NaT,True,False
1,client_fef1a8f436438636,content_0081d123d9e489eb,2026-03-02,7.083333,12,0,7.812500,client_fef1a8f436438636,keyword_b8e40434e7f63eab,url_ad7808313ab6a14c,...,3,2025-12-26,None,gpt-4o-mini,9519,1567,NaT,NaT,True,False
2,client_fef1a8f436438636,content_0081d123d9e489eb,2026-03-03,5.444444,9,0,7.447917,client_fef1a8f436438636,keyword_b8e40434e7f63eab,url_ad7808313ab6a14c,...,3,2025-12-26,None,gpt-4o-mini,9519,1567,NaT,NaT,True,False
3,client_fef1a8f436438636,content_0081d123d9e489eb,2026-03-04,5.500000,14,0,6.780093,client_fef1a8f436438636,keyword_b8e40434e7f63eab,url_ad7808313ab6a14c,...,3,2025-12-26,None,gpt-4o-mini,9519,1567,NaT,NaT,True,False
4,client_fef1a8f436438636,content_0081d123d9e489eb,2026-03-05,4.466667,15,0,6.460069,client_fef1a8f436438636,keyword_b8e40434e7f63eab,url_ad7808313ab6a14c,...,3,2025-12-26,None,gpt-4o-mini,9519,1567,NaT,NaT,True,False


**Feature 1 -- `gsc_avg_position` (that day):** knowable at the decision moment because it's
Google's placement of the page, set by the ranking algorithm independently of whether anyone
clicks that day -- it doesn't derive from the day's own click count.

**Feature 2 -- `prior_7d_avg_position`:** knowable because the window function explicitly
excludes the current day (`1 PRECEDING`) -- everything it uses happened strictly before the row
being scored.

**Feature 3 -- content age at `report_date`** (content's creation date vs. `report_date`, from
`dim_content`): knowable because a page's publish date predates every day it's ever reported on,
by construction.

**Feature 4 -- content type / category** (from `dim_content`, exact column name confirmed by the
`DESCRIBE` in Section 1): knowable because it's a static attribute set when the page was
published, long before any given `report_date`.

**Feature 5 -- word count / length attribute** (from `dim_content`, exact column name confirmed
above): knowable for the same reason as Feature 4 -- it's fixed at publish/last-edit time, not
something that changes with that day's traffic.


## 5. The trap -- add one label-derived column on purpose

`ctr` is computed from `gsc_clicks / gsc_impressions`. If `gsc_clicks` itself is added back in as
a "feature," any model can trivially reconstruct the label -- that's leakage, not a discovery.


In [13]:
%pip install --upgrade scikit-learn pyarrow

^C
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


  Using cached pyarrow-25.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 451.0 kB/s eta 0:00:17
   --- ------------------------------------ 0.8/8.2 MB 588.6 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/8.2 MB 588.6 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/8.2 MB 588.6 kB/s eta 0:00:13
   ----- ---------------------------------- 1.0/8.2 MB 551.3 kB/s eta 0:00:13
   ----- ---------------------------------- 1.0/8.2 M

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

honest = feature_frame.dropna(subset=["gsc_avg_position", "prior_7d_avg_position"]).copy()
honest["ctr"] = honest["gsc_clicks"] / honest["gsc_impressions"]

honest_features = ["gsc_avg_position", "prior_7d_avg_position"]  # + confirmed dim_content cols once named
X = honest[honest_features]
y = honest["ctr"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
honest_score = r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
print(f"HONEST held-out R^2 (position-only features): {honest_score:.3f}")

# --- now the trap: add a column derived straight from the label ---
honest["LEAK_same_day_clicks"] = honest["gsc_clicks"]  # this literally IS the label's numerator
leaky_features = honest_features + ["LEAK_same_day_clicks"]
X_leak = honest[leaky_features]
X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y, test_size=0.25, random_state=42)
leaky_score = r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
print(f"LEAKY held-out R^2 (+ gsc_clicks as a 'feature'): {leaky_score:.3f}  <- jumps toward 1.0, not real signal")

# --- delete the leak, keep the honest number ---
del honest["LEAK_same_day_clicks"]
print(f"\nKept: honest R^2 = {honest_score:.3f}. Deleted: the leaky column and its score.")


HONEST held-out R^2 (position-only features): 0.005
LEAKY held-out R^2 (+ gsc_clicks as a 'feature'): 0.050  <- jumps toward 1.0, not real signal

Kept: honest R^2 = 0.005. Deleted: the leaky column and its score.


## 6. Data limits

**Named limitation:** daily-grain CTR is noisy even after an impressions floor. A single
content-day with, say, 8 impressions can swing from 0% to 25% CTR on one click -- the >=5
filter above keeps out the worst of it, but doesn't make a single day's CTR trustworthy on its
own. Any real scoring pass in this lane should aggregate CTR over a rolling window (7 or 30
days) per content item before comparing it to a position-tier baseline, not compare single days
to each other. This contract's daily grain is the right grain for *building* features (position,
recency) but the wrong grain for the *label* itself.

A second, related limit: this is one month (March 2026) of an unbalanced panel -- client history
depth varies a lot, so even within March, some clients contribute far more content-days than
others. Nothing here should be read as "the average page," only as "pages from the clients with
March history."


In [15]:
# No computation needed -- documenting the limitation named above.
limitation = (
    "Daily CTR is too noisy at single-day grain even after an impressions floor; "
    "needs a rolling window before being used as a trustworthy label."
)
print(limitation)


Daily CTR is too noisy at single-day grain even after an impressions floor; needs a rolling window before being used as a trustworthy label.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.